# TODO

In [1]:
ENCODERS = [
    # "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    # "intfloat/e5-base",
]

ENCODE_BATCH_SIZE = 32

AE_EPOCH_N = 5
AE_BATCH_SIZE = 8192

device = "mps"

In [2]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import re
import random
from tqdm import tqdm
from itertools import product

import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/artfultom/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [5]:
filenames = [
    "../datasets/mistral_essays_1.csv",
    "../datasets/mistral_essays_2.csv",
    "../datasets/mistral_essays_3.csv",
    "../datasets/mistral_essays_4.csv",
    "../datasets/mistral_essays_5.csv",
    "../datasets/llama_essays_1.csv",
    "../datasets/llama_essays_2.csv",
    "../datasets/llama_essays_3.csv",
    "../datasets/llama_essays_4.csv",
    "../datasets/llama_essays_5.csv",
    "../datasets/deepseek_essays_1.csv",
    "../datasets/deepseek_essays_2.csv",
    "../datasets/deepseek_essays_3.csv",
    "../datasets/deepseek_essays_4.csv",
    "../datasets/deepseek_essays_5.csv",
    "../datasets/chatgpt_essays_1.csv",
    "../datasets/chatgpt_essays_2.csv",
    "../datasets/chatgpt_essays_3.csv",
    "../datasets/chatgpt_essays_4.csv",
    "../datasets/chatgpt_essays_5.csv",
]
df_ai = pd.concat([pd.read_csv(f) for f in filenames], ignore_index=True).assign(label=1)[["text", "label"]]#.sample(n=50, random_state=SEED)

df_human = pd.read_csv("../datasets/ivy_panda_essays.csv").assign(label=0)[["text", "label"]].reset_index(drop=True)#.sample(n=100, random_state=SEED)
df_train, df_human_test = train_test_split(df_human, test_size=len(df_ai), random_state=SEED)

df_all = pd.concat([df_ai, df_human_test], ignore_index=True).sample(frac=1, random_state=SEED)

In [6]:
print(f"Загружено {len(df_train)} записей Ivy Panda (живые, train)")
print(f"Загружено {len(df_all)} записей (синтетика + живые, test)")

Загружено 90805 записей Ivy Panda (живые, train)
Загружено 74976 записей (синтетика + живые, test)


In [7]:
class AE(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, dim),
        )
    def forward(self, x):
        return self.dec(self.enc(x))

In [8]:
def print_big_header(text, width=100):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)

In [9]:
def normalize_f(data):
    data = torch.from_numpy(data).float()
    data = F.normalize(data, p=2, dim=0)
    return data.numpy()

def dwp_pooling_diag_far(X, eps=1e-8):
    mu = X.mean(axis=0)
    var = X.var(axis=0) + eps
    dists = np.sqrt(((X - mu) ** 2 / var).sum(axis=1))
    
    weights = np.exp(dists)
    weights /= weights.sum() + eps

    return weights @ X

def get_raw_embeddings(model, data, normalize_embeddings=False):
    all_sentences = []
    spans = []

    for text in tqdm(data):
        text = re.sub(r"\n+", ". ", text)
        sentences = nltk.sent_tokenize(text)

        start = len(all_sentences)
        all_sentences.extend(sentences)
        end = len(all_sentences)

        spans.append((start, end))

    all_embeddings = model.encode(
        all_sentences,
        batch_size=ENCODE_BATCH_SIZE,
        normalize_embeddings=normalize_embeddings,
        show_progress_bar=True,
        device=device,
    )

    return spans, all_embeddings

def get_embeddings(spans, all_embeddings):
    mean_vals = []

    max_vals = []
    mean_diff_vals = []
    med_diff_vals = []
    var_diff_vals = []
    dwp_vals = []

    for start, end in tqdm(spans):
        embeddings = all_embeddings[start:end]

        mean_val = np.mean(embeddings, axis=0)

        max_val = np.max(embeddings, axis=0)
        max_val = normalize_f(max_val)

        diffs = embeddings[1:] - embeddings[:-1]
        
        mean_diff_val = np.mean(diffs, axis=0)
        mean_diff_val = normalize_f(mean_diff_val)

        med_diff_val = np.median(diffs, axis=0)
        med_diff_val = normalize_f(med_diff_val)

        var_diff_val = np.var(diffs, axis=0)
        var_diff_val = normalize_f(var_diff_val)

        dwp_val = dwp_pooling_diag_far(embeddings)

        mean_vals.append(mean_val)
        max_vals.append(max_val)
        mean_diff_vals.append(mean_diff_val)
        med_diff_vals.append(med_diff_val)
        var_diff_vals.append(var_diff_val)
        dwp_vals.append(dwp_val)

    return (
        mean_vals,

        max_vals,
        mean_diff_vals,
        med_diff_vals,
        var_diff_vals,
        dwp_vals,
    )

In [10]:
def ae_score(X_train, X_test):
    d = X_train.shape[1]
    
    ae = AE(d).to(device)
    ae.train()
    
    opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    Xtr = torch.tensor(X_train, dtype=torch.float32)
    dataset = torch.utils.data.TensorDataset(Xtr)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=AE_BATCH_SIZE,
        shuffle=True,
        drop_last=False
    )

    for epoch in range(AE_EPOCH_N):
        for (batch,) in loader:
            batch = batch.to(device)

            out = ae(batch)
            loss = loss_fn(out, batch)

            opt.zero_grad()
            loss.backward()
            opt.step()

    ae.eval()
    Xe = torch.tensor(X_test, dtype=torch.float32).to(device)

    with torch.no_grad():
        recon = ae(Xe).cpu().numpy()

    return np.mean((X_test - recon) ** 2, axis=1)

In [11]:
t_values = [0.0, 0.4, 0.8, 1.2, 1.6, 2]

results = {}
for model_name in ENCODERS:
    print_big_header(f"МОДЕЛЬ: {model_name}")
    
    model = SentenceTransformer(model_name)

    spans_train, all_embeddings_train = get_raw_embeddings(model, df_train['text'].tolist(), True)
    spans_test, all_embeddings_test = get_raw_embeddings(model, df_all['text'].tolist(), True)

    train_embeddings = list(get_embeddings(spans_train, all_embeddings_train))
    test_embeddings = list(get_embeddings(spans_test, all_embeddings_test))

    if model_name not in results:
        results[model_name] = {}

    c = len(train_embeddings)
    for t in product(t_values, repeat=c - 1):
        t = (1,) + t
        
        X_train = np.array(train_embeddings[0])
        X_test = np.array(test_embeddings[0])

        for i in range(1, len(t)):
            if t[i] == 0:
                continue

            X_train = np.concatenate(
                [X_train, np.array(train_embeddings[i]) * t[i]], axis=1
            )
            X_test = np.concatenate(
                [X_test, np.array(test_embeddings[i]) * t[i]], axis=1
            )

        y_test = df_all['label'].values.astype(int)

        # 1. Autoencoder (MLP)
        ae_scores = ae_score(X_train, X_test)

        results[model_name][t] = {
            "Autoencoder": (ae_scores, y_test),
        }

        print(f"Модель {model_name}, t={t} - DONE")


                                     МОДЕЛЬ: all-mpnet-base-v2                                      


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████████████████████████████████████████████████████████████████████| 90805/90805 [01:05<00:00, 1389.93it/s]


Batches:   0%|          | 0/228612 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████| 74976/74976 [00:40<00:00, 1831.19it/s]


Batches:   0%|          | 0/138968 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████| 74976/74976 [01:02<00:00, 1207.95it/s]


Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.0, 0.0) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.0, 0.4) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.0, 0.8) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.0, 1.2) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.0, 1.6) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.0, 2) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.4, 0.0) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.4, 0.4) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.4, 0.8) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.4, 1.2) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.4, 1.6) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.4, 2) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.8, 0.0) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.8, 0.4) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.0, 0.8, 0.8) - DONE
Модель all-mpnet-base-v2, t=(1, 0.0, 0.0, 0.

In [12]:
best_per_encoder = {}
for model_name, t_dict in results.items():
    best_per_encoder[model_name] = {}

    for t, methods in t_dict.items():
        for method_name, (scores, y_true) in methods.items():
            auc_score = roc_auc_score(y_true, scores)
            ap_score = average_precision_score(y_true, scores)

            if method_name not in best_per_encoder[model_name]:
                best_per_encoder[model_name][method_name] = {
                    "best_auc": auc_score,
                    "ap": ap_score,
                    "best_t": t
                }
            else:
                if auc_score > best_per_encoder[model_name][method_name]["best_auc"]:
                    best_per_encoder[model_name][method_name] = {
                        "best_auc": auc_score,
                        "ap": ap_score,
                        "best_t": t
                    }

In [13]:
top_k = 10
top_per_encoder = {}
for model_name, t_dict in results.items():
    all_scores = {
        "Autoencoder": []
    }

    for t, methods in t_dict.items():
        for method_name in ["Autoencoder"]:
            scores, y_true = methods[method_name]

            auc_score = roc_auc_score(y_true, scores)

            all_scores[method_name].append({
                "t": t,
                "auc": auc_score
            })

    top_per_encoder[model_name] = {}

    for method_name in ["Autoencoder"]:
        sorted_scores = sorted(
            all_scores[method_name],
            key=lambda x: x["auc"],
            reverse=True
        )

        top_per_encoder[model_name][method_name] = sorted_scores[:top_k]

In [14]:
for model_name, methods in top_per_encoder.items():
    print_big_header(f"Энкодер: {model_name}")

    for method_name, top_list in methods.items():
        print(f"\n{method_name} — TOP {top_k} по ROC-AUC:")

        for i, info in enumerate(top_list, 1):
            print(f"{i}. ROC-AUC = {info['auc']:.4f} | t = {info['t']}")


                                     Энкодер: all-mpnet-base-v2                                     

Autoencoder — TOP 10 по ROC-AUC:
1. ROC-AUC = 0.9242 | t = (1, 0.0, 1.2, 0.4, 1.2, 0.0)
2. ROC-AUC = 0.9241 | t = (1, 0.0, 1.2, 0.8, 0.8, 0.0)
3. ROC-AUC = 0.9236 | t = (1, 0.4, 1.2, 0.0, 1.2, 0.0)
4. ROC-AUC = 0.9235 | t = (1, 0.0, 1.2, 0.0, 0.0, 0.0)
5. ROC-AUC = 0.9234 | t = (1, 0.4, 1.2, 0.4, 0.8, 0.0)
6. ROC-AUC = 0.9233 | t = (1, 0.0, 1.2, 1.2, 1.2, 0.0)
7. ROC-AUC = 0.9233 | t = (1, 0.0, 1.2, 0.8, 1.2, 0.0)
8. ROC-AUC = 0.9233 | t = (1, 0.0, 1.2, 1.2, 0.8, 0.0)
9. ROC-AUC = 0.9231 | t = (1, 0.4, 1.2, 0.0, 0.8, 0.0)
10. ROC-AUC = 0.9229 | t = (1, 0.4, 1.2, 0.8, 0.8, 0.0)


## Вывод

TODO
